# Time-Series Feature Engineering for PD Model
## Rolling Window Features from Transaction Data

This notebook builds comprehensive rolling time-series features from transaction-level data using Polars.

**Features Generated:**
- Rolling window aggregations (7d, 30d, 90d, 180d)
- Momentum features (percent change vs lag dates)
- Temporal features (day of week, season, etc.)
- Coefficient of variation (volatility measures)

**Output:**
- `df_train_features`: Training features with rolling windows
- `df_holdout_features`: Holdout features with rolling windows
- `df_train_merged`: Complete training dataset (features + account + labels)
- `df_holdout_merged`: Complete holdout dataset (features + account)

In [ ]:
# Import libraries
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')
print(f'Polars version: {pl.__version__}')

## Step 1: Load Data

In [ ]:
# Load training data
train_tx = pl.read_csv('/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_tx.csv')
train_account = pl.read_csv('/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_account.csv')
train_label = pl.read_csv('/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_label.csv')

print('✓ Training data loaded')
print(f'  - Transactions: {train_tx.shape[0]:,} rows × {train_tx.shape[1]} cols')
print(f'  - Accounts: {train_account.shape[0]:,} rows × {train_account.shape[1]} cols')
print(f'  - Labels: {train_label.shape[0]:,} rows × {train_label.shape[1]} cols')

# Load holdout data
holdout_tx = pl.read_csv('/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_holdout_tx.csv')
holdout_account = pl.read_csv('/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_holdout_account.csv')

print('\n✓ Holdout data loaded')
print(f'  - Transactions: {holdout_tx.shape[0]:,} rows × {holdout_tx.shape[1]} cols')
print(f'  - Accounts: {holdout_account.shape[0]:,} rows × {holdout_account.shape[1]} cols')

In [ ]:
# Inspect data structure
print('Training Transaction Data Sample:')
print(train_tx.head(3))
print('\nColumn Types:')
print(train_tx.dtypes)

In [ ]:
print('Account Data Sample:')
print(train_account.head(3))
print('\nLabel Data Sample:')
print(train_label.head(3))

## Step 2: Standardize DataFrames

In [ ]:
# Build rolling time-series features by Restaurant_ID and snapshot Tx_date
LABEL_KEY = 'Restaurant_ID'

# Standardize dataframe names with df_ prefix
df_train_account = train_account.clone()
df_train_tx = train_tx.clone()
df_train_label = train_label.clone()
df_holdout_account = holdout_account.clone()
df_holdout_tx = holdout_tx.clone()

print('✓ All dataframes standardized with df_ prefix')
print(f'  - Label key: {LABEL_KEY}')

## Step 3: Define Time-Series Feature Engineering Function

In [ ]:
def _build_timeseries_features(df_tx, windows=(7, 30, 90, 180)):
    """
    Build rolling time-series features from transaction data.
    
    Parameters:
    - df_tx: Polars DataFrame with transaction data
    - windows: Tuple of rolling window sizes (in days)
    
    Returns:
    - Polars DataFrame with rolling features and temporal features
    """
    
    # Step 1: Prepare transaction data
    df_work = (
        df_tx
        .select([LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours'])
        .with_columns(pl.col('Tx_date').str.to_date(strict=False))
        .sort([LABEL_KEY, 'Tx_date'])
    )

    # Get unique snapshots (Restaurant_ID, Tx_date combinations)
    df_snapshots = df_work.select([LABEL_KEY, 'Tx_date']).unique().sort([LABEL_KEY, 'Tx_date'])

    print(f'  - Total snapshots: {df_snapshots.shape[0]:,}')
    print(f'  - Unique restaurants: {df_snapshots[LABEL_KEY].n_unique()}')

    # Step 2: Build rolling window aggregations
    for window in windows:
        df_window = (
            df_work
            .group_by_dynamic(
                index_column='Tx_date',
                group_by=LABEL_KEY,
                every='1d',
                period=f'{window}d',
                closed='right',
            )
            .agg([
                pl.col('processing_volume').mean().alias('avg_proc_vol'),
                pl.col('processing_volume').min().alias('min_proc_vol'),
                pl.col('processing_volume').max().alias('max_proc_vol'),
                pl.col('processing_volume').std().alias('std_proc_vol'),
                pl.col('Tx_hours').mean().alias('avg_tx_hours'),
                pl.col('Tx_hours').min().alias('min_tx_hours'),
                pl.col('Tx_hours').max().alias('max_tx_hours'),
                pl.col('Tx_hours').std().alias('std_tx_hours'),
            ])
            .with_columns([
                pl.when(pl.col('avg_proc_vol') != 0)
                .then(pl.col('std_proc_vol') / pl.col('avg_proc_vol'))
                .otherwise(None)
                .alias(f'cv_proc_vol_{window}d'),
                pl.when(pl.col('avg_tx_hours') != 0)
                .then(pl.col('std_tx_hours') / pl.col('avg_tx_hours'))
                .otherwise(None)
                .alias(f'cv_tx_hours_{window}d'),
            ])
            .rename({
                'avg_proc_vol': f'avg_proc_vol_{window}d',
                'min_proc_vol': f'min_proc_vol_{window}d',
                'max_proc_vol': f'max_proc_vol_{window}d',
                'avg_tx_hours': f'avg_tx_hours_{window}d',
                'min_tx_hours': f'min_tx_hours_{window}d',
                'max_tx_hours': f'max_tx_hours_{window}d',
            })
            .select([
                LABEL_KEY,
                'Tx_date',
                f'avg_proc_vol_{window}d',
                f'min_proc_vol_{window}d',
                f'max_proc_vol_{window}d',
                f'avg_tx_hours_{window}d',
                f'min_tx_hours_{window}d',
                f'max_tx_hours_{window}d',
                f'cv_proc_vol_{window}d',
                f'cv_tx_hours_{window}d',
            ])
        )

        df_snapshots = df_snapshots.join(df_window, on=[LABEL_KEY, 'Tx_date'], how='left')

    # Step 3: Add momentum features (percent change vs lag dates)
    df_daily = (
        df_work
        .select([LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours'])
        .unique()
        .rename({
            'processing_volume': 'curr_processing_volume',
            'Tx_hours': 'curr_tx_hours',
        })
    )
    df_snapshots = df_snapshots.join(df_daily, on=[LABEL_KEY, 'Tx_date'], how='left')

    for lag_days in windows:
        lag_df = (
            df_daily
            .with_columns((pl.col('Tx_date') + pl.duration(days=lag_days)).alias('Tx_date'))
            .rename({
                'curr_processing_volume': f'proc_vol_{lag_days}d_ago',
                'curr_tx_hours': f'tx_hours_{lag_days}d_ago',
            })
        )

        df_snapshots = df_snapshots.join(lag_df, on=[LABEL_KEY, 'Tx_date'], how='left')

        df_snapshots = df_snapshots.with_columns([
            pl.when(pl.col(f'proc_vol_{lag_days}d_ago') != 0)
            .then(
                (pl.col('curr_processing_volume') - pl.col(f'proc_vol_{lag_days}d_ago'))
                / pl.col(f'proc_vol_{lag_days}d_ago')
            )
            .otherwise(None)
            .alias(f'pct_change_proc_vol_vs_{lag_days}d_ago'),
            pl.when(pl.col(f'tx_hours_{lag_days}d_ago') != 0)
            .then(
                (pl.col('curr_tx_hours') - pl.col(f'tx_hours_{lag_days}d_ago'))
                / pl.col(f'tx_hours_{lag_days}d_ago')
            )
            .otherwise(None)
            .alias(f'pct_change_tx_hours_vs_{lag_days}d_ago'),
        ])

    # Step 4: Add temporal features
    df_snapshots = df_snapshots.with_columns([
        pl.col('Tx_date').dt.weekday().alias('snapshot_day_of_week'),
        pl.col('Tx_date').dt.ordinal_day().alias('snapshot_day_of_year'),
        pl.col('Tx_date').dt.month().alias('snapshot_month'),
        pl.col('Tx_date').dt.quarter().alias('snapshot_quarter'),
    ])

    return df_snapshots.sort([LABEL_KEY, 'Tx_date'])


print('✓ Time-series feature engineering function defined')

## Step 4: Build Training Features

In [ ]:
print('Building training features...')
df_train_features = _build_timeseries_features(df_train_tx)
print('✓ Training features built successfully')

In [ ]:
print('Training Features Shape:', df_train_features.shape)
print('\nColumns:', df_train_features.columns)
print(f'\nTotal features generated: {len(df_train_features.columns) - 2}')  # -2 for Restaurant_ID and Tx_date

In [ ]:
print('Training Features Sample:')
df_train_features.head(3)

## Step 5: Build Holdout Features

In [ ]:
print('Building holdout features...')
df_holdout_features = _build_timeseries_features(df_holdout_tx)
print('✓ Holdout features built successfully')

In [ ]:
print('Holdout Features Shape:', df_holdout_features.shape)
print('\nColumns match:', df_holdout_features.columns == df_train_features.columns)

In [ ]:
print('Holdout Features Sample:')
df_holdout_features.head(3)

## Step 6: Clean Data (Remove Index Artifacts)

In [ ]:
# Remove source file index artifacts before merge (avoids Unnamed: 0_x / Unnamed: 0_y)
df_train_account_clean = df_train_account.select([c for c in df_train_account.columns if not c.startswith('Unnamed:')])
df_train_label_clean = df_train_label.select([c for c in df_train_label.columns if not c.startswith('Unnamed:')])
df_holdout_account_clean = df_holdout_account.select([c for c in df_holdout_account.columns if not c.startswith('Unnamed:')])

print('✓ Unnamed columns cleaned')
print(f'  - Train account cols: {len(df_train_account_clean.columns)}')
print(f'  - Train label cols: {len(df_train_label_clean.columns)}')
print(f'  - Holdout account cols: {len(df_holdout_account_clean.columns)}')

## Step 7: Merge Datasets

In [ ]:
# Merge requested datasets
# train = features + account + labels
# holdout = features + account

print('Merging training dataset...')
df_train_merged = (
    df_train_features
    .join(df_train_account_clean, on=LABEL_KEY, how='left')
    .join(df_train_label_clean, on=LABEL_KEY, how='left')
)
print('✓ Training dataset merged')

print('\nMerging holdout dataset...')
df_holdout_merged = df_holdout_features.join(df_holdout_account_clean, on=LABEL_KEY, how='left')
print('✓ Holdout dataset merged')

## Step 8: Verify Output Datasets

In [ ]:
print('='*80)
print('FEATURE ENGINEERING SUMMARY')
print('='*80)

print(f"\n{'Dataset':<25} {'Rows':>12} {'Columns':>12} {'Description'}")
print("-" * 80)
print(f"{'df_train_features':<25} {df_train_features.shape[0]:>12,} {df_train_features.shape[1]:>12}  Rolling window features + temporal")
print(f"{'df_holdout_features':<25} {df_holdout_features.shape[0]:>12,} {df_holdout_features.shape[1]:>12}  Rolling window features + temporal")
print(f"{'df_train_merged':<25} {df_train_merged.shape[0]:>12,} {df_train_merged.shape[1]:>12}  Features + account + labels")
print(f"{'df_holdout_merged':<25} {df_holdout_merged.shape[0]:>12,} {df_holdout_merged.shape[1]:>12}  Features + account")

print("\n" + "="*80)

In [ ]:
print('Any Unnamed columns (train)?', any(c.startswith('Unnamed:') for c in df_train_merged.columns))
print('Any Unnamed columns (holdout)?', any(c.startswith('Unnamed:') for c in df_holdout_merged.columns))

print('\n✓ All data cleaning verified')

## Step 9: Feature Statistics

In [ ]:
print('FEATURE CATEGORIES')
print('='*80)

# Count feature categories
all_cols = df_train_merged.columns
rolling_features = [c for c in all_cols if any(f'{w}d' in c for w in [7, 30, 90, 180])]
momentum_features = [c for c in all_cols if 'pct_change' in c]
temporal_features = [c for c in all_cols if 'snapshot' in c]
current_features = [c for c in all_cols if c.startswith('curr_')]
lag_features = [c for c in all_cols if 'ago' in c and 'pct_change' not in c]
account_features = [c for c in df_train_account_clean.columns if c != LABEL_KEY]
label_features = [c for c in df_train_label_clean.columns if c != LABEL_KEY]

print(f"\nRolling Window Features (7d, 30d, 90d, 180d): {len(rolling_features)}")
print(f"  Sample: {rolling_features[:3]}")

print(f"\nMomentum Features (pct_change): {len(momentum_features)}")
print(f"  Sample: {momentum_features[:2]}")

print(f"\nTemporal Features (snapshot): {len(temporal_features)}")
print(f"  Features: {temporal_features}")

print(f"\nCurrent Day Features: {len(current_features)}")
print(f"  Features: {current_features}")

print(f"\nLag Features (N days ago): {len(lag_features)}")
print(f"  Sample: {lag_features[:2]}")

print(f"\nAccount Features: {len(account_features)}")
print(f"  Features: {account_features}")

print(f"\nLabel Features: {len(label_features)}")
print(f"  Features: {label_features}")

print(f"\nTotal Features (excluding keys): {len(all_cols) - 1}")

## Step 10: Data Quality Checks

In [ ]:
print('DATA QUALITY CHECKS')
print('='*80)

print('\nMissing Values in Training Set:')
missing_counts = df_train_merged.null_count()
missing_df = pd.DataFrame({
    'Column': missing_counts.columns,
    'Missing_Count': missing_counts.row(0),
    'Missing_Pct': [100 * c / df_train_merged.shape[0] for c in missing_counts.row(0)]
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Pct', ascending=False)
print(missing_df.head(10).to_string(index=False))

if len(missing_df) == 0:
    print('  No missing values found!')

In [ ]:
print('\nTarget Variable Distribution (Training Set):')
if 'loan_default' in df_train_merged.columns:
    target_dist = df_train_merged.select('loan_default').group_by('loan_default').count().sort('loan_default')
    print(target_dist)
    
    total = df_train_merged.shape[0]
    defaults = df_train_merged.filter(pl.col('loan_default') == 1).shape[0]
    default_rate = 100 * defaults / total
    
    print(f'\nTotal Restaurants: {total:,}')
    print(f'Defaults: {defaults:,}')
    print(f'Default Rate: {default_rate:.2f}%')
else:
    print('  Target variable not found in training set')

## Step 11: Export Processed Data

In [ ]:
import os
from pathlib import Path

# Create output directory
output_dir = '/Users/pradark/Documents/011. Work/Toast/Principal Data Scientist Capital Case Study/polars_EBT/feature_engineering_output'
Path(output_dir).mkdir(parents=True, exist_ok=True)

print('Exporting processed datasets...')

# Export to CSV
df_train_features.write_csv(f'{output_dir}/train_features_timeseries.csv')
print(f'✓ Exported: train_features_timeseries.csv')

df_holdout_features.write_csv(f'{output_dir}/holdout_features_timeseries.csv')
print(f'✓ Exported: holdout_features_timeseries.csv')

df_train_merged.write_csv(f'{output_dir}/train_merged_complete.csv')
print(f'✓ Exported: train_merged_complete.csv')

df_holdout_merged.write_csv(f'{output_dir}/holdout_merged_complete.csv')
print(f'✓ Exported: holdout_merged_complete.csv')

# Export to Parquet (more efficient)
df_train_merged.write_parquet(f'{output_dir}/train_merged_complete.parquet')
print(f'✓ Exported: train_merged_complete.parquet')

df_holdout_merged.write_parquet(f'{output_dir}/holdout_merged_complete.parquet')
print(f'✓ Exported: holdout_merged_complete.parquet')

print(f'\n✓ All datasets exported to: {output_dir}')

## Step 12: Final Summary

In [ ]:
print('\n' + '='*80)
print('TIME-SERIES FEATURE ENGINEERING COMPLETE')
print('='*80)

print(f"""
INPUT DATASETS:
  - Training Transactions: {train_tx.shape[0]:,} rows
  - Training Accounts: {train_account.shape[0]:,} rows
  - Training Labels: {train_label.shape[0]:,} rows
  - Holdout Transactions: {holdout_tx.shape[0]:,} rows
  - Holdout Accounts: {holdout_account.shape[0]:,} rows

OUTPUT DATASETS:
  - Train Features: {df_train_features.shape[0]:,} rows × {df_train_features.shape[1]} cols
  - Holdout Features: {df_holdout_features.shape[0]:,} rows × {df_holdout_features.shape[1]} cols
  - Train Merged: {df_train_merged.shape[0]:,} rows × {df_train_merged.shape[1]} cols
  - Holdout Merged: {df_holdout_merged.shape[0]:,} rows × {df_holdout_merged.shape[1]} cols

FEATURES GENERATED:
  - Rolling Window (7d, 30d, 90d, 180d): {len(rolling_features)} features
  - Momentum (pct_change vs lag): {len(momentum_features)} features
  - Temporal (day of week, season): {len(temporal_features)} features
  - Current Day: {len(current_features)} features
  - Lag Features: {len(lag_features)} features
  - Account Features: {len(account_features)} features
  - Total: {len(all_cols) - 1} input features

OUTPUT FORMAT:
  - CSV format (human-readable)
  - Parquet format (efficient for modeling)
  - Location: {output_dir}

NEXT STEPS:
  1. Use df_train_merged and df_holdout_merged for modeling
  2. Features are ready for PD model training
  3. All rolling windows properly aligned by date
  4. No data leakage (features use historical data only)
""")

print('='*80)